# 99 — Complete Workflow Lab

**Related guide:** [99_COMPLETE_REFERENCE_APPLICATION.md](../docs/guides/99_COMPLETE_REFERENCE_APPLICATION.md)

**Related script:** [complete/data_pipeline.py](../examples/complete/data_pipeline.py)

This notebook follows the DX05 loop: **Concept → Construction → Inspection → Modification → Execution → Observation → Experiment**.

## Construction

Assemble the complete `fetch → validate → transform → publish` workflow with durable SQLite metadata.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from pyworkflowkit._compat.v1_root import (
    RunContext,
    RuntimeSettings,
    TaskHandle,
    TaskId,
    WorkflowRuntime,
    task,
    workflow,
)


@task
def fetch() -> list[int]:
    return [1, 2, 3]


@task(depends_on=(fetch,))
def validate(context: RunContext) -> list[int]:
    values = context.dependency_outputs[TaskId("fetch")]
    return [int(value) for value in values]


@task(depends_on=(validate,))
def transform(context: RunContext) -> list[int]:
    values = context.dependency_outputs[TaskId("validate")]
    return [int(value) * 10 for value in values]


@task(depends_on=(transform,))
def publish(context: RunContext) -> int:
    values = context.dependency_outputs[TaskId("transform")]
    return len(values)


@workflow(id="notebooks.complete", version="1")
def pipeline() -> tuple[TaskHandle, ...]:
    return (fetch, validate, transform, publish)

In [ ]:
with TemporaryDirectory() as directory:
    workspace = Path(directory)
    settings = RuntimeSettings.load(
        overrides={
            "runtime": {"workspace": workspace},
            "metadata": {
                "backend": "sqlite",
                "sqlite_path": "runtime.sqlite3",
                "sqlite_wal": False,
            },
        }
    )
    runtime = WorkflowRuntime(settings)
    for handle in pipeline.task_handles():
        runtime.register(handle.handler_ref, handle.handler)
    definition = pipeline.build()
    run = runtime.run(definition)
    manifest = runtime.manifest(definition, run.run_id)
    lineage = runtime.lineage(definition, run.run_id)
    print(
        {
            "status": run.status.value,
            "tasks": len(manifest.tasks),
            "events": len(manifest.events),
            "lineage_tasks": len(lineage.tasks),
        }
    )

## Experiment

Add a branch, retry policy, or external workload. Predict the evidence changes before rerunning the lab.